# Pipeline with mT5 FOL Model


In [ ]:
# Prover9 setup
from pathlib import Path

PROVER9_SRC = Path("/kaggle/input/datasets/baohg153/prover9/Prover9-LADR-2026-4F")
PROVER9_DST = Path("/kaggle/working/Prover9-LADR-2026-4F")

if Path("/kaggle/working").exists() and PROVER9_SRC.exists() and not PROVER9_DST.exists():
    !cp -r /kaggle/input/datasets/baohg153/prover9/Prover9-LADR-2026-4F /kaggle/working/
    %cd /kaggle/working/Prover9-LADR-2026-4F
    !make all STATIC=1
    %cd /kaggle/working/
else:
    print("Skipping Prover9 setup. Configure Pipeline.engine path before full run if needed.")


In [ ]:
!pip install -q -U pandas nltk regex torch datasets transformers accelerate sentencepiece tqdm

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json
from pathlib import Path

import torch

In [ ]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        item["fol_premises"] = item.get("fol_premises", None)
        item["fol_conclusion"] = item.get("fol_conclusion", None)
        return {
            "id": item["id"],
            "nat_premises": item["nat_premises"],
            "fol_premises":  "" if item["fol_premises"] is None else item["fol_premises"],
            "nat_conclusion": item["nat_conclusion"],
            "fol_conclusion": "" if item["fol_conclusion"] is None else item["fol_conclusion"],
            "label": item["label"]
        }

In [ ]:
import gc
import inspect
import os
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def _as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(str(item).strip() for item in value if str(item).strip())
    return str(value).strip()


def _first_present(item: Dict[str, Any], keys: Sequence[str]) -> Any:
    for key in keys:
        if key in item:
            return item[key]
    return None


class MT5FOLModel:
    """Seq2Seq mT5 adapter with the same interface used by final-pipeline.ipynb.

    The existing pipeline calls only prepare_sample(), train(), predict(), and saves
    model/tokenizer through .model, .tokenizer, and .adapter_dir. This class keeps
    that surface area while using encoder-decoder labels instead of causal-LM
    prompt masking.
    """

    def __init__(
        self,
        model_name_or_path: str,
        output_dir: str = "/kaggle/working/mt5_fol_model",
        save_dir: str | None = None,
        max_source_length: int = 768,
        max_target_length: int = 768,
        generation_max_new_tokens: int = 512,
        local_files_only: bool = False,
        device: str | None = None,
        fp16: bool = True,
    ):
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        self.model_name_or_path = model_name_or_path
        self.model_name = model_name_or_path
        self.output_dir = output_dir
        self.save_dir = save_dir or output_dir
        self.adapter_dir = self.save_dir
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
        self.generation_max_new_tokens = generation_max_new_tokens
        self.local_files_only = local_files_only
        self.fp16 = fp16

        if device is None:
            if torch.cuda.is_available():
                device = "cuda"
            elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
                device = "mps"
            else:
                device = "cpu"
        self.device = torch.device(device)

        print(f"Loading mT5 tokenizer from {model_name_or_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name_or_path,
            local_files_only=local_files_only,
        )

        model_kwargs: Dict[str, Any] = {"local_files_only": local_files_only}
        if self.device.type == "cuda" and fp16:
            model_kwargs["torch_dtype"] = torch.float16

        print(f"Loading mT5 seq2seq model from {model_name_or_path}")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path, **model_kwargs)
        self._ensure_special_tokens()
        self.model.to(self.device)

    def _ensure_special_tokens(self) -> None:
        if self.tokenizer.pad_token is None and self.tokenizer.eos_token is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        if self.model.config.pad_token_id is None and self.tokenizer.pad_token_id is not None:
            self.model.config.pad_token_id = self.tokenizer.pad_token_id
        if self.model.config.decoder_start_token_id is None:
            if self.model.config.pad_token_id is not None:
                self.model.config.decoder_start_token_id = self.model.config.pad_token_id
            elif self.tokenizer.pad_token_id is not None:
                self.model.config.decoder_start_token_id = self.tokenizer.pad_token_id

    def _normalized_fields(self, data_example: Dict[str, Any]) -> Dict[str, str]:
        return {
            "id": _as_text(data_example.get("id") or data_example.get("story_id") or ""),
            "nat_premises": _as_text(
                _first_present(data_example, ("nat_premises", "nat_premise", "premises"))
            ),
            "nat_conclusion": _as_text(_first_present(data_example, ("nat_conclusion", "conclusion"))),
            "fol_premises": _as_text(_first_present(data_example, ("fol_premises", "fol_premise"))),
            "fol_conclusion": _as_text(data_example.get("fol_conclusion")),
            "label": _as_text(data_example.get("label")),
        }

    def build_source(self, data_example: Dict[str, Any]) -> str:
        example = self._normalized_fields(data_example)
        return (
            "translate natural language to first-order logic:\n"
            "Premises:\n"
            f"{example['nat_premises']}\n"
            "Conclusion:\n"
            f"{example['nat_conclusion']}"
        )

    def build_target(self, data_example: Dict[str, Any]) -> str:
        example = self._normalized_fields(data_example)
        return f"{example['fol_premises']}\n{example['fol_conclusion']}".strip()

    def prepare_sample(self, data_example: Dict[str, Any]) -> Dict[str, List[int]]:
        source = self.build_source(data_example)
        target = self.build_target(data_example)
        if not target:
            example_id = data_example.get("id", "<unknown>")
            raise ValueError(f"Example {example_id!r} has no FOL target for mT5 training.")

        model_inputs = self.tokenizer(
            source,
            padding="max_length",
            truncation=True,
            max_length=self.max_source_length,
        )
        try:
            labels = self.tokenizer(
                text_target=target,
                padding="max_length",
                truncation=True,
                max_length=self.max_target_length,
            )
        except TypeError:
            with self.tokenizer.as_target_tokenizer():
                labels = self.tokenizer(
                    target,
                    padding="max_length",
                    truncation=True,
                    max_length=self.max_target_length,
                )

        pad_id = self.tokenizer.pad_token_id
        label_ids = labels["input_ids"]
        if pad_id is not None:
            label_ids = [token if token != pad_id else -100 for token in label_ids]

        return {
            "input_ids": model_inputs["input_ids"],
            "attention_mask": model_inputs["attention_mask"],
            "labels": label_ids,
        }

    def train(
        self,
        train_data: Any,
        epochs: int = 1,
        batch_size: int = 1,
        learning_rate: float = 5e-5,
        gradient_accumulation_steps: int = 8,
    ) -> None:
        import torch
        from datasets import Dataset
        from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

        rows = getattr(train_data, "data", train_data)
        rows = list(rows)
        if not rows:
            raise ValueError("No mT5 training rows were provided.")

        dataset = Dataset.from_list(rows)
        dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

        if self.device.type == "cuda":
            self.model.config.use_cache = False

        candidates: Dict[str, Any] = {
            "output_dir": self.output_dir,
            "per_device_train_batch_size": batch_size,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "num_train_epochs": epochs,
            "learning_rate": learning_rate,
            "logging_steps": 10,
            "report_to": "none",
            "save_total_limit": 1,
            "fp16": bool(self.fp16 and self.device.type == "cuda"),
            "dataloader_pin_memory": False,
            "predict_with_generate": False,
            "optim": "adafactor",
        }
        arg_signature = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
        training_kwargs = {key: value for key, value in candidates.items() if key in arg_signature}

        collator = DataCollatorForSeq2Seq(
            tokenizer=self.tokenizer,
            model=self.model,
            label_pad_token_id=-100,
        )

        trainer_candidates: Dict[str, Any] = {
            "model": self.model,
            "args": Seq2SeqTrainingArguments(**training_kwargs),
            "train_dataset": dataset,
            "data_collator": collator,
        }
        trainer_signature = inspect.signature(Seq2SeqTrainer.__init__).parameters
        if "processing_class" in trainer_signature:
            trainer_candidates["processing_class"] = self.tokenizer
        elif "tokenizer" in trainer_signature:
            trainer_candidates["tokenizer"] = self.tokenizer

        trainer = Seq2SeqTrainer(
            **{key: value for key, value in trainer_candidates.items() if key in trainer_signature}
        )
        trainer.train()
        self.save()

        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def predict(self, dataset: Any, max_new_tokens: int = 768, num_outputs: int = 10) -> List[Dict[str, Any]]:
        import torch

        max_new_tokens = max_new_tokens or self.generation_max_new_tokens
        self.model.eval()
        results: List[Dict[str, Any]] = []

        for index in range(len(dataset)):
            raw_item = dataset[index]
            item = self._normalized_fields(raw_item)
            source = self.build_source(raw_item)
            inputs = self.tokenizer(
                source,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.max_source_length,
            )
            inputs = {key: value.to(self.device) for key, value in inputs.items()}

            generation_kwargs = {
                "max_new_tokens": max_new_tokens,
                "pad_token_id": self.tokenizer.pad_token_id,
                "num_return_sequences": max(1, num_outputs),
                "do_sample": False,
            }
            if num_outputs > 1:
                generation_kwargs["num_beams"] = max(1, num_outputs)
                generation_kwargs["early_stopping"] = True
            else:
                generation_kwargs["num_beams"] = 1

            with torch.no_grad():
                output_ids = self.model.generate(**inputs, **generation_kwargs)

            decoded_outputs = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            generated_fols = [text.strip() for text in decoded_outputs[: max(1, num_outputs)]]

            results.append(
                {
                    "id": item["id"],
                    "natural": f"{item['nat_premises']}\n{item['nat_conclusion']}",
                    "predictions": [{"fol": fol, "label": None} for fol in generated_fols],
                    "label": item["label"],
                }
            )

        return results

    def save(self, path: Optional[str] = None) -> None:
        save_path = Path(path or self.save_dir)
        save_path.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(save_path)
        self.tokenizer.save_pretrained(save_path)

In [ ]:
# mT5 checkpoint configuration.
MT5_CHECKPOINT_PATH = "/kaggle/working/outputs/basic2_mt5_base_folio/checkpoint-1080"

fol_model = MT5FOLModel(
    model_name_or_path=MT5_CHECKPOINT_PATH,
    output_dir="/kaggle/working/mt5_pipeline_model",
    save_dir="/kaggle/working/mt5_pipeline_model",
    max_source_length=768,
    max_target_length=768,
    generation_max_new_tokens=512,
    local_files_only=True,
)

In [ ]:
VALID_PATH_CANDIDATES = [
    Path("dataset/filtered/valid/folio_valid.json"),
    Path("/kaggle/input/neurosymbolic-reasoning/dataset/filtered/valid/folio_valid.json"),
    Path("/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/folio_valid.json"),
]

valid_path = next((p for p in VALID_PATH_CANDIDATES if p.exists()), None)
if valid_path is None:
    raise FileNotFoundError("Could not find a FOLIO validation file for the smoke test.")

with open(valid_path, "r", encoding="utf-8") as f:
    valid_rows = json.load(f)[:2]

smoke_model = fol_model
smoke_dataset = ManualDataset(valid_rows)
prepared = smoke_model.prepare_sample(valid_rows[0])
print("prepare_sample keys:", sorted(prepared.keys()))
print("prepare_sample lengths:", {k: len(v) for k, v in prepared.items() if isinstance(v, list)})
print("non-ignored label tokens:", sum(1 for token in prepared["labels"] if token != -100))

smoke_predictions = smoke_model.predict(smoke_dataset, max_new_tokens=64, num_outputs=1)
print(json.dumps(smoke_predictions[0], ensure_ascii=False, indent=2)[:2000])

required = {"id", "natural", "predictions", "label"}
assert all(required <= set(row) for row in smoke_predictions)
assert all(isinstance(row["predictions"], list) and row["predictions"] for row in smoke_predictions)
assert all("fol" in row["predictions"][0] for row in smoke_predictions)
print("mT5 notebook smoke schema passed.")

In [ ]:
import copy
from collections import Counter

import numpy as np
import nltk
from nltk.sem.logic import LogicParser
from nltk.inference import Prover9
import regex as re

In [ ]:
class Engine:
    def __init__(self, prover9_path='/kaggle/working/Prover9-LADR-2026-4F/bin'):
        self.parser = LogicParser()
        self.prover9_path = prover9_path
        self.prover = Prover9()
        self.prover.config_prover9(prover9_path)

    def _transform_xor(self, text):
        """
        Transform A ⊕ B to ¬(A ⟷ B)
        """
        while '⊕' in text:
            idx = text.find('⊕')
            
            # Find left component
            lhs_end = idx
            while lhs_end > 0 and text[lhs_end-1].isspace():
                lhs_end -= 1
                
            curr = lhs_end - 1
    
            if text[curr] == ')':
                balance = 0
                while curr >= 0:
                    if text[curr] == ')': balance += 1
                    elif text[curr] == '(': balance -= 1
                    curr -= 1
                    if balance == 0: break
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
            else:
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
                
            lhs = text[lhs_start:lhs_end]
    
            # Find right component
            rhs_start = idx + 1
            while rhs_start < len(text) and text[rhs_start].isspace():
                rhs_start += 1
                
            curr = rhs_start
            if text[curr].isalnum() or text[curr] == '_':
                while curr < len(text) and (text[curr].isalnum() or text[curr] == '_'):
                    curr += 1
                if curr < len(text) and text[curr] == '(':
                    balance = 0
                    while curr < len(text):
                        if text[curr] == '(': balance += 1
                        elif text[curr] == ')': balance -= 1
                        curr += 1
                        if balance == 0: break
                rhs_end = curr
            elif text[curr] == '(':
                balance = 0
                while curr < len(text):
                    if text[curr] == '(': balance += 1
                    elif text[curr] == ')': balance -= 1
                    curr += 1
                    if balance == 0: break
                rhs_end = curr
            else:
                rhs_end = curr + 1
                
            rhs = text[rhs_start:rhs_end]
    
            # Transform XOR
            new_expression = f"¬({lhs} ⟷ {rhs})"
            text = text[:lhs_start] + new_expression + text[rhs_end:]
    
        return text
    
    def _translate_fol(self, fol_text: str):
        # Transform XOR
        fol_text = self._transform_xor(fol_text)
        
        # '-' --> '_'
        fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

        replacements = {
            '∀': 'all ', 
            '∃': 'exists ',
            '∧': '&', 
            '∨': '|',
            '⊕': '^',
            '¬': '-',
            '→': '->', 
            '⟷': '<->',
            '↔': '<->'
        }
        for k, v in replacements.items():
            fol_text = fol_text.replace(k, v)

        # Add dot: "all x (P(x))" --> "all x. (P(x))"
        fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

        # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
        words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
        reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
        
        for w in set(words):
            if w not in reserved_words:
                fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
        return fol_text


    def _is_valid_fol(self, fol_list):
        try:
            for line in fol_list:
                if line.strip():
                    self.parser.parse(line)
            return True
        except Exception:
            return False

    def _check_conclusion(self, premises, conclusion):
        '''
        This function is the engine. It checks whether the conclusion is True/False/Uncertain based on the list of premises.
        Args:
            premises: list[str]
            conclusion: str
        Returns: 
            str ("True"/"False"/"Uncertain")
        '''
        # Read fol strings
        translated_premises = [self._translate_fol(p) for p in premises]
        translated_conclusion = self._translate_fol(conclusion)

        if (not self._is_valid_fol(translated_premises) or not self._is_valid_fol([translated_conclusion])):
            error_msg = f"Invalid FOL syntax detected!"
            raise ValueError(error_msg)
        
        try:
            parsed_premises = [self.parser.parse(p) for p in translated_premises]
            parsed_conclusion = self.parser.parse(translated_conclusion)
        except Exception as e:
            raise f"Error: {e}"

        # Check conclusion
        if self.prover.prove(parsed_conclusion, []):
            raise Exception("Error: The conclusion itself is True!")
        elif self.prover.prove(parsed_conclusion.negate(), []):
            raise Exception("Error: The conclusion itself is False!")
        elif not self.prover.prove(None, parsed_premises):
            is_true = self.prover.prove(parsed_conclusion, parsed_premises)
            if is_true:
                return "True"

            is_false = self.prover.prove(parsed_conclusion.negate(), parsed_premises)
            if is_false:
                return "False"

            return "Uncertain"
        else: 
            raise Exception("Error: The premises are inconsistent!")
    
    def check_conclusion(self, data):
        fol_list = data["predictions"]
        new_fol_list = []
        for fol in fol_list:
            try:
                premises = fol["fol"].split("\n")[:-1]
                conclusion = fol["fol"].split("\n")[-1]
                result = self._check_conclusion(premises, conclusion)

                if result == "True":
                    fol["label"] = "True"
                elif result == "False":   # FIX 2
                    fol["label"] = "False"
                else:
                    fol["label"] = "Uncertain"
                new_fol_list.append(fol)
            except Exception as e:
                # print(e)
                continue

        data["predictions"] = new_fol_list
        return data

In [ ]:
def lv_distance(s1, s2):
    if len(s1) < len(s2):
        return lv_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

class DataFilter:
    def __init__(self):
        self.duplicate_count = 0
        self.syntax_error_count = 0
        self.length_ratio_count = 0
    
    def _is_valid_syntax(self, fol_str: str) -> bool:
        if not fol_str:
            return False
            
        stack = []
        matching_bracket = {')': '(', ']': '[', '}': '{'}
        
        for char in fol_str:                   
            if char in matching_bracket.values():
                stack.append(char)
            elif char in matching_bracket.keys():
                if not stack or stack.pop() != matching_bracket[char]:
                    return False

        return len(stack) == 0
    
    
    def _is_valid_length_ratio(self, nat_str: str, fol_str: str, lowerbound_ratio: float=0.5, upperbound_ratio: float=2.0) -> bool:
        nat_length = len(nat_str)
        if nat_length == 0:
            return False
    
        fol_length = len(fol_str)
        ratio = fol_length / nat_length
        
        return lowerbound_ratio <= ratio <= upperbound_ratio


    def _update_names(self, fol, ratio_threshold=1/3):
        pattern = r'(\w+)\('
        matches = re.findall(pattern, fol)
    
        counts = Counter(matches)
        all_names = list(set(matches))
    
        single_names = [name for name, count in counts.items() if count == 1]
    
        update_names = {}
    
        for n1 in single_names:
            for n2 in all_names:
                if n2 in single_names and len(n2) > len(n1):
                    continue
                if n1 != n2 and lv_distance(n1, n2) <= ratio_threshold * max(len(n1), len(n2)):
                    update_names[n1] = n2
    
        return update_names
    
    
    def filter(self, entry: dict, lowerbound_ratio:float=0.5, upperbound_ratio:float=2.0) -> dict:
        valid_predictions = []
        nat_str = entry.get("natural", "")
        
        natural_text = entry.get("natural", "").strip()
        sentences = [s for s in natural_text.split('.') if s.strip()]
        nat_length = len(sentences)

        for pred in entry.get("predictions", []):
            fol = pred.get("fol", "")
            
            # Validate syntax (bracket balancing)
            if not self._is_valid_syntax(fol):
                self.syntax_error_count += 1
                continue
            
            premise_list = [p.strip() for p in fol.split('\n') if p.strip()]
            conclusion = premise_list[-1]
            
            premise_list = list(dict.fromkeys(premise_list[:-1]))

            # if len(premise_list) != (nat_length - 1):
            #     self.duplicate_count += 1
            #     continue
            
            # Check for length ratio
            if not self._is_valid_length_ratio(nat_str, fol, lowerbound_ratio, upperbound_ratio):
                self.length_ratio_count += 1
                continue

            valid_fol = "\n".join(premise_list + [conclusion])
            update = self._update_names(valid_fol)
            for key, value in update.items():
                pattern = fr'(?<![a-zA-Z0-9]){key}\('
                valid_fol = re.sub(pattern, value + "(", valid_fol)

            valid_predictions.append({'fol': valid_fol, 'label': None})
            
        filtered_element = entry.copy()
        filtered_element["predictions"] = valid_predictions

        return filtered_element

def filter2_pred(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        return True
    return False

def filter2(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        positive_sample = [prediction for prediction in fol_predictions if prediction["label"] == ground_truth]
        negative_sample = [prediction for prediction in fol_predictions if prediction["label"] != ground_truth]
        return positive_sample, negative_sample

    return None

In [ ]:
class Pipeline:
    def __init__(self, fol_model):
        self.fol_model = fol_model
        self.filter_1 = DataFilter()
        self.engine = Engine(prover9_path="/kaggle/working/Prover9-LADR-2026-4F/bin")
    
    def get_data(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            dataset = json.load(f)
        return dataset

    def __save_file_prediction(self, predictions, file_name="predictions.json"):
        with open(file_name, "w", encoding="utf-8") as f:
            json.dump(predictions, f, indent=4, ensure_ascii=False)
            
    def train(self, train_set_path,
              valid_set_path,
              silver_set_path,
              new_train_set_path,
              folio_set_path,
              batch_pop=32,
              max_steps=3, batchsize=4, previous_step=1):

        best_accuracy = 0.0

        train_set = self.get_data(train_set_path)
        valid_set = self.get_data(valid_set_path)
        
        finetune_set = []
        silver_set = []

        folio_set = self.get_data(folio_set_path)
        for i in range(len(folio_set)):
            processed_tensor_sample = self.fol_model.prepare_sample(folio_set[i])
            finetune_set.append({
                "item": processed_tensor_sample,
                "usage_count": 0
            })
            silver_set.append(folio_set[i])

        print(f"Bắt đầu quy trình tự học với tổng {len(train_set)} mẫu trong dataset gốc.")

        for step in range(max_steps):
            print(f"\n========== STEP {step + 1 + previous_step}/{max_steps + previous_step} ==========")

            if len(train_set) == 0:
                print("Đã lấy hết dữ liệu từ train_set ban đầu. Tiếp theo chỉ thực hiện finetune.")
            else:
                # Lấy batchsize mẫu từ trên xuống và xóa khỏi train_set
                current_batch_size = min(batch_pop, len(train_set))
                batch_data = train_set[:current_batch_size]
                train_set = train_set[current_batch_size:]
    
                # Mô hình dự đoán
                batch_data = ManualDataset(batch_data)
                predicted_batch = self.fol_model.predict(batch_data)
                # predicted_batch = self.format_predict_dataset(batch_data)
    
                for original_data, pred_data in zip(batch_data, predicted_batch):
                    is_success = False
                    
                    pred_data = self.filter_1.filter(pred_data)
                    pred_data = self.engine.check_conclusion(pred_data)
                    
                    result_filter2 = filter2(pred_data['predictions'], pred_data["label"])
                    if result_filter2 is not None:
                        positive_fol, _ = result_filter2
                        chosen_pred = positive_fol[0]
    
                        original_nat_premises = original_data.get("nat_premises", "")
                        original_nat_conclusion = original_data.get("nat_conclusion", "")
                        if not original_nat_premises or not original_nat_conclusion:
                            natural_parts = pred_data.get("natural", "").rsplit("\n", 1)
                            if len(natural_parts) == 2:
                                fallback_premises, fallback_conclusion = natural_parts
                            else:
                                fallback_premises, fallback_conclusion = pred_data.get("natural", ""), ""
                            original_nat_premises = original_nat_premises or fallback_premises
                            original_nat_conclusion = original_nat_conclusion or fallback_conclusion
    
                        formatted_sample = {
                            "id": pred_data["id"],
                            "nat_premises": original_nat_premises,
                            "nat_conclusion": original_nat_conclusion,
                            "fol_premises": '\n'.join(chosen_pred['fol'].split('\n')[:-1]),
                            "fol_conclusion": chosen_pred['fol'].split('\n')[-1],
                            "label": pred_data["label"]
                        }
    
                        silver_set.append(formatted_sample)
    
                        processed_tensor_sample = self.fol_model.prepare_sample(formatted_sample)
                        finetune_set.append({
                            "item": processed_tensor_sample, # Đã là input_ids, attention_mask, labels
                            "usage_count": 0
                        })
                        
                        is_success = True
    
                    if not is_success:
                        train_set.append(original_data)
    
                print(f"Kích thước finetune_set hiện tại: {len(finetune_set)}")
    
                # Huấn luyện bằng dữ liệu từ finetune_set
                if len(finetune_set) >= batch_pop:
                    weights = np.array([1.0 / (sample["usage_count"] + 1) for sample in finetune_set])
                    probs = weights / weights.sum() # Chuẩn hóa để tổng xác suất = 1.0
    
                    sampled_indices = np.random.choice(
                        len(finetune_set), 
                        size=batch_pop, 
                        replace=False, 
                        p=probs
                    )
    
                    train_batch = []
                    for idx in sampled_indices:
                        finetune_set[idx]["usage_count"] += 1
                        train_batch.append(finetune_set[idx]["item"])
    
                    # print(f"Bắt đầu huấn luyện mô hình với {batchsize} mẫu từ finetune_set...")
                    train_batch = ManualDataset(train_batch)
                    self.fol_model.train(
                        train_data=train_batch,
                        batch_size=batchsize
                    )
                else:
                    print("Chưa đủ dữ liệu trong finetune_set để tạo batch huấn luyện. Chuyển sang step tiếp theo...")

            # 50 steps valid một lần, cũng như lưu silver_data và new_train_set một lần
            if step % 50 == 0:
                if len(silver_set) > 0:
                    with open(silver_set_path, "w", encoding="utf-8") as f:
                        json.dump(silver_set, f, ensure_ascii=False, indent=4)
                    print(f"Đã lưu/cập nhật {len(silver_set)} mẫu vào {silver_set_path}")

                with open(new_train_set_path, "w", encoding="utf-8") as f:
                    json.dump(train_set, f, ensure_ascii=False, indent=4)
                print(f"Đã cập nhật new_train_set vào {new_train_set_path}")
                
                # Valid thủ công ở đây
                count_correct = 0
                valid_set = ManualDataset(valid_set)
                predictions = self.fol_model.predict(valid_set, num_outputs=10)
                # predictions = self.format_predict_dataset(valid_set, num_outputs=10)
                # self.__save_file_prediction(predictions)
                filtered_predictions = []
                for prediction in predictions:
                    prediction = self.filter_1.filter(prediction)
                    prediction = self.engine.check_conclusion(prediction)
                    filtered_predictions.append(prediction)
                    count_correct += (filter2(prediction['predictions'], prediction["label"]) is not None)
                self.__save_file_prediction(filtered_predictions)
                accuracy = count_correct / len(predictions)
                print("Validation accuracy:", accuracy)
                if accuracy > best_accuracy or accuracy == 1:
                    self.fol_model.model.save_pretrained(self.fol_model.adapter_dir)
                    self.fol_model.tokenizer.save_pretrained(self.fol_model.adapter_dir)
                    best_accuracy = accuracy

In [ ]:
# Pipeline wiring. This can fail if Prover9 is not configured; the mT5 smoke test above does not need it.
try:
    pipeline = Pipeline(fol_model=fol_model)
except Exception as exc:
    pipeline = None
    print(f"Pipeline initialization skipped or failed. Check Prover9 before full run: {exc}")

In [ ]:
# Optional mini-pipeline smoke test. Keep disabled unless the model smoke test passes.
RUN_MINI_PIPELINE_SMOKE = False

if RUN_MINI_PIPELINE_SMOKE:
    if pipeline is None:
        print("Skipping mini-pipeline smoke because Pipeline could not initialize. Check Prover9.")
    else:
        try:
            tmp_dir = Path("/kaggle/working/mt5_mini_pipeline_smoke") if Path("/kaggle/working").exists() else Path("outputs/mt5_mini_pipeline_smoke")
            tmp_dir.mkdir(parents=True, exist_ok=True)
            with open(valid_path, "r", encoding="utf-8") as f:
                rows = json.load(f)[:2]
            train_path = tmp_dir / "train_2.json"
            valid_path_small = tmp_dir / "valid_2.json"
            folio_path = tmp_dir / "folio_2.json"
            train_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
            valid_path_small.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
            folio_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
            pipeline.train(
                train_set_path=str(train_path),
                valid_set_path=str(valid_path_small),
                silver_set_path=str(tmp_dir / "silver_data.json"),
                new_train_set_path=str(tmp_dir / "new_train_set.json"),
                folio_set_path=str(folio_path),
                batch_pop=1,
                max_steps=1,
                batchsize=1,
                previous_step=0,
            )
        except Exception as exc:
            print(f"Mini-pipeline smoke skipped or failed gracefully: {exc}")

In [ ]:
# Full pipeline call template. Do not run until smoke tests pass and Prover9 is configured.
# pipeline.train(
#     train_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/train_without_fol.json",
#     valid_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/valid_without_fol.json",
#     silver_set_path="/kaggle/working/silver_data.json",
#     new_train_set_path="/kaggle/working/new_train_set.json",
#     folio_set_path="/kaggle/input/datasets/ductri0981/neurosymbolic-dataset/folio_train.json",
#     batch_pop=32,
#     max_steps=201,
#     batchsize=1,
#     previous_step=0,
# )
